# Petrobras Financial Statement Validator

Este projeto analisa as demonstrações financeiras da Petrobras
com dados públicos da CVM, aplicando Python, SQL e auditoria
contábil para validar consistência e detectar anomalias.

## Objetivos

- Validar consistência contábil
- Detectar anomalias trimestrais
- Consultar dados com SQL
- Visualizar KPIs financeiros com Python (matplotlib)

In [7]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt

## 1. Leitura e filtragem dos dados

Nesta etapa, carregamos os dados de demonstrações financeiras da CVM e filtramos apenas os registros da Petrobras.

Também restringimos a análise para os anos de 2023 e 2024, tornando o projeto mais objetivo e focado.

In [ ]:
df = pd.read_csv(
    "../data/raw/dfp_cia_aberta_DRE_con.csv",
    sep=";",
    encoding="latin1"
    decimal=","
)

petrobras = df[df["DENOM_CIA"].str.contains("PETROBRAS", case=False, na=False)].copy()

petrobras["DT_REFER"] = pd.to_datetime(petrobras["DT_REFER"])
petrobras = petrobras[petrobras["DT_REFER"].dt.year.isin([2023, 2024])]

petrobras[["DT_REFER", "DS_CONTA", "VL_CONTA"]].head()

,DT_REFER,DS_CONTA,VL_CONTA
19968,2023-12-31,Receita de Venda de Bens e/ou Serviços,641256000.0
19969,2023-12-31,Receita de Venda de Bens e/ou Serviços,511994000.0
19970,2023-12-31,Custo dos Bens e/ou Serviços Vendidos,-307156000.0
19971,2023-12-31,Custo dos Bens e/ou Serviços Vendidos,-242061000.0
19972,2023-12-31,Resultado Bruto,334100000.0


## 2. Seleção das contas relevantes
Selecionamos as contas-chave para análise: Receita líquida, Lucro líquido, Caixa e equivalentes, Empréstimos e Patrimônio líquido.

In [9]:
selected_accounts = [
    "Receita líquida",
    "Lucro líquido",
    "Caixa e equivalentes",
    "Empréstimos",
    "Patrimônio líquido"
]

df_clean = petrobras[petrobras["DS_CONTA"].isin(selected_accounts)].copy()
df_clean.head()

,CNPJ_CIA,DT_REFER,VERSAO,DENOM_CIA,CD_CVM,GRUPO_DFP,MOEDA,ESCALA_MOEDA,ORDEM_EXERC,DT_INI_EXERC,DT_FIM_EXERC,CD_CONTA,DS_CONTA,VL_CONTA,ST_CONTA_FIXA


## 3. Padronização dos dados
Renomeamos as colunas para nomes mais intuitivos.

In [10]:
df_clean = df_clean.rename(columns={
    "DT_REFER": "date",
    "DS_CONTA": "account",
    "VL_CONTA": "value"
})

## 4. Armazenamento em banco de dados (SQLite)
Os dados tratados são armazenados em um banco SQLite, permitindo consultas SQL para análise posterior.

In [11]:
conn = sqlite3.connect("../data/processed/petrobras.db")
df_clean.to_sql("financial_statements", conn, if_exists="replace", index=False)
print("Dados carregados com sucesso no SQLite.")

Dados carregados com sucesso no SQLite.


## 5. Análise da receita com SQL
Consultamos a evolução da receita líquida ao longo do tempo, garantindo ordenação temporal correta.

In [12]:
query = """
SELECT date, value
FROM financial_statements
WHERE account = 'Receita líquida'
ORDER BY date
"""
revenue = pd.read_sql(query, conn)
revenue.head()

,date,value


## 6. Cálculo do nível de endividamento (Debt Ratio)
Debt Ratio = Empréstimos / Patrimônio Líquido — calculado via JOIN em SQL, com proteção contra divisão por zero.

In [13]:
query = """
SELECT
    a.date,
    a.value AS debt,
    b.value AS equity,
    a.value * 1.0 / b.value AS debt_ratio
FROM financial_statements a
JOIN financial_statements b
    ON a.date = b.date
WHERE a.account = 'Empréstimos'
  AND b.account = 'Patrimônio líquido'
  AND b.value != 0
"""
debt_ratio = pd.read_sql(query, conn)

avg_debt = debt_ratio["debt_ratio"].mean()
print(f"Debt ratio médio: {avg_debt:.2f}")
debt_ratio.head()

Debt ratio médio: nan


,date,debt,equity,debt_ratio


## 7. Visualizações principais
Analisamos a evolução da receita líquida e do debt ratio ao longo do tempo.

In [14]:
ax = revenue.plot(
    x="date",
    y="value",
    figsize=(12, 5),
    marker="o",
    grid=True,
    title="Evolução da Receita Líquida - Petrobras"
)
ax.set_xlabel("Período")
ax.set_ylabel("Valor (R$ milhões)")  # ← unidade explícita
plt.tight_layout()
plt.show()

ax = debt_ratio.plot(
    x="date",
    y="debt_ratio",
    figsize=(12, 5),
    marker="o",
    grid=True,
    title="Debt Ratio ao longo do tempo"
)
ax.set_xlabel("Período")
ax.set_ylabel("Debt Ratio (Empréstimos / PL)")
plt.tight_layout()
plt.show()

TypeError: no numeric data to plot

## 8. Cálculo de variação trimestral (QoQ)
Calculamos a variação percentual entre períodos consecutivos, garantindo ordenação temporal antes do cálculo.

In [ ]:
# Garante ordenação antes de calcular variação — pct_change depende da ordem
revenue = revenue.sort_values("date").reset_index(drop=True).copy()
revenue["qoq"] = revenue["value"].pct_change()

max_growth = revenue["qoq"].max()
print(f"Maior crescimento trimestral: {max_growth:.2%}")

## 9. Identificação de anomalias
Variações superiores a 25% entre períodos consecutivos são sinalizadas como anomalias para investigação.

In [ ]:
ANOMALY_THRESHOLD = 0.25  # constante nomeada — fácil de ajustar

alerts = revenue[revenue["qoq"].abs() > ANOMALY_THRESHOLD].copy()
print(f"{len(alerts)} anomalia(s) identificada(s) (variação > {ANOMALY_THRESHOLD:.0%})")
alerts.style.format({"qoq": "{:.2%}"})

## 10. Detecção estatística de outliers (Z-score)
Aplicamos o Z-score para identificar valores estatisticamente distantes da média. Pontos além de ±2 desvios merecem investigação.

In [ ]:
ZSCORE_THRESHOLD = 2.0  # constante nomeada

revenue["zscore"] = (
    revenue["value"] - revenue["value"].mean()
) / revenue["value"].std()

ax = revenue.plot(
    x="date",
    y="zscore",
    figsize=(12, 5),
    marker="o",
    grid=True,
    title="Z-score da Receita Líquida"
)
ax.axhline(ZSCORE_THRESHOLD, linestyle="--", color="red", label=f"+{ZSCORE_THRESHOLD}")
ax.axhline(-ZSCORE_THRESHOLD, linestyle="--", color="red", label=f"-{ZSCORE_THRESHOLD}")
ax.set_xlabel("Período")
ax.set_ylabel("Z-score")
ax.legend()
plt.tight_layout()
plt.show()

## 11. Exportação dos dados processados

Exportamos os dados tratados para CSV como camada de persistência,
permitindo reuso em análises futuras sem necessidade de reprocessar
os dados brutos da CVM.

In [ ]:
df_clean.to_csv("../data/processed/final_data.csv", index=False)
revenue.to_csv("../data/processed/revenue_analysis.csv", index=False)
alerts.to_csv("../data/processed/alerts.csv", index=False)
print("Arquivos exportados com sucesso.")

## 12. Conclusão

A análise validou a consistência dos dados financeiros da Petrobras
e identificou variações relevantes nos principais indicadores.

O uso combinado de Python, SQL e análise exploratória replicou
processos comuns em auditoria e análise financeira, com visualizações
produzidas inteiramente em Python.

Pipeline concluído:
dados brutos → tratamento → SQL → análise → alertas → visualização → dashboard executivo → SQL analytics

## 13. Executive Dashboard 

A seguir, consolidamos os principais indicadores do projeto em um painel executivo construído diretamente no notebook.

O objetivo é facilitar a interpretação dos resultados e comunicar os principais insights de negócio.

### 13.1 KPI Panel

Os principais indicadores são resumidos abaixo para visão rápida da performance financeira.

In [ ]:
# ── KPIs base ──────────────────────────────────────────────────────────────
query_kpi = """
SELECT account, date, value
FROM financial_statements
WHERE account IN ('Receita líquida', 'Lucro líquido')
ORDER BY account, date
"""
df_kpi = pd.read_sql(query_kpi, conn)

# Último e penúltimo período por conta
def get_kpi(account):
    s = df_kpi[df_kpi["account"] == account]["value"]
    last  = s.iloc[-1]
    prev  = s.iloc[-2]
    delta = (last - prev) / abs(prev)
    return last, delta

receita_val,  receita_delta  = get_kpi("Receita líquida")
lucro_val,    lucro_delta    = get_kpi("Lucro líquido")

# Margem líquida — KPI novo
df_pivot = df_kpi.pivot_table(index="date", columns="account", values="value")
df_pivot["margem_liquida"] = df_pivot["Lucro líquido"] / df_pivot["Receita líquida"]
margem_val   = df_pivot["margem_liquida"].iloc[-1]
margem_delta = df_pivot["margem_liquida"].iloc[-1] - df_pivot["margem_liquida"].iloc[-2]

debt_val = avg_debt  # calculado na seção 6

# ── Formatação dos deltas com cor ANSI ─────────────────────────────────────
def fmt_delta(d, is_pp=False):
    sinal  = "▲" if d >= 0 else "▼"
    cor    = "\033[32m" if d >= 0 else "\033[31m"
    reset  = "\033[0m"
    sufixo = "pp" if is_pp else "%"
    return f"{cor}{sinal} {abs(d)*100:.1f}{sufixo}{reset}"

# ── Impressão do painel ─────────────────────────────────────────────────────
SEP = "─" * 64

print(SEP)
print(f"  PETROBRAS — EXECUTIVE KPI PANEL   2023–2024")
print(SEP)
print(f"  {'Receita Líquida':<22} R$ {receita_val/1e9:>8.1f} bn    {fmt_delta(receita_delta)}")
print(f"  {'Lucro Líquido':<22} R$ {lucro_val/1e9:>8.1f} bn    {fmt_delta(lucro_delta)}")
print(f"  {'Margem Líquida':<22}    {margem_val*100:>8.1f} %     {fmt_delta(margem_delta, is_pp=True)}")
print(f"  {'Debt Ratio Médio':<22}    {debt_val:>8.2f}       — referência setor: 1.5")
print(SEP)

### 13.2 Painel de Gráficos

Evolução temporal dos quatro indicadores principais em layout de relatório executivo.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle(
    "Petrobras — Indicadores Financeiros 2023–2024",
    fontsize=13, fontweight="normal", y=1.01
)

STYLE      = dict(color="black", linewidth=1.5, marker="o", markersize=4)
MEAN_STYLE = dict(color="gray", linewidth=1, linestyle="--")
dates_rev  = df_pivot.index
dates_dbt  = debt_ratio["date"]

# ── Subplot 1: Receita líquida ──────────────────────────────────────────────
ax = axes[0, 0]
vals = df_pivot["Receita líquida"]
ax.plot(dates_rev, vals / 1e9, **STYLE)
ax.axhline(vals.mean() / 1e9, **MEAN_STYLE, label="média")
pico_idx = vals.idxmax()
ax.annotate(
    f"Pico\nR${vals[pico_idx]/1e9:.0f}bn",
    xy=(pico_idx, vals[pico_idx] / 1e9),
    xytext=(10, -20), textcoords="offset points",
    fontsize=8, color="gray"
)
ax.set_title("Receita Líquida (R$ bn)", fontsize=10, fontweight="normal")
ax.tick_params(labelsize=8)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(fontsize=8)

# ── Subplot 2: Lucro líquido ────────────────────────────────────────────────
ax = axes[0, 1]
vals = df_pivot["Lucro líquido"]
ax.plot(dates_rev, vals / 1e9, **STYLE)
ax.axhline(vals.mean() / 1e9, **MEAN_STYLE, label="média")
ax.set_title("Lucro Líquido (R$ bn)", fontsize=10, fontweight="normal")
ax.tick_params(labelsize=8)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(fontsize=8)

# ── Subplot 3: Margem líquida ───────────────────────────────────────────────
ax = axes[1, 0]
vals = df_pivot["margem_liquida"] * 100
ax.plot(dates_rev, vals, **STYLE)
ax.axhline(vals.mean(), **MEAN_STYLE, label="média")
ax.set_title("Margem Líquida (%)", fontsize=10, fontweight="normal")
ax.tick_params(labelsize=8)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(fontsize=8)

# ── Subplot 4: Debt ratio ───────────────────────────────────────────────────
ax = axes[1, 1]
ax.plot(dates_dbt, debt_ratio["debt_ratio"], **STYLE)
ax.axhspan(0, 1.5, alpha=0.06, color="green", label="zona segura (< 1.5)")
ax.axhline(1.5, color="gray", linewidth=0.8, linestyle=":")
ax.set_title("Debt Ratio", fontsize=10, fontweight="normal")
ax.tick_params(labelsize=8)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(fontsize=8)

fig.autofmt_xdate(rotation=30)
plt.tight_layout()
plt.savefig("../data/processed/dashboard.png", dpi=150, bbox_inches="tight")
plt.show()

### 13.3 Radar de Anomalias

Consolidação dos alertas QoQ e outliers Z-score identificados ao longo da análise.

In [ ]:
outliers = revenue[revenue["zscore"].abs() > ZSCORE_THRESHOLD][["date", "value", "zscore"]].copy()
outliers["tipo"] = "Z-score"
outliers = outliers.rename(columns={"zscore": "indicador"})

alertas_qoq = alerts[["date", "value", "qoq"]].copy()
alertas_qoq["tipo"] = "QoQ"
alertas_qoq = alertas_qoq.rename(columns={"qoq": "indicador"})

radar = pd.concat([alertas_qoq, outliers], ignore_index=True).sort_values("date")

n_alertas = len(radar)
status    = "ATENÇÃO — anomalias detectadas" if n_alertas > 0 else "OK — nenhuma anomalia crítica"

print(f"\n  STATUS AUDITORIA: {status}")
print(f"  Total de eventos sinalizados: {n_alertas}\n")

if n_alertas > 0:
    display(
        radar
        .style
        .format({"value": "R$ {:.1f}bn", "indicador": "{:.2%}"})
        .apply(
            lambda col: [
                "background-color: #fff3cd" if v == "QoQ"
                else "background-color: #f8d7da"
                for v in col
            ],
            subset=["tipo"]
        )
        .set_caption("Anomalias identificadas — Receita Líquida 2023–2024")
    )
else:
    print("  Nenhum evento para exibir.")

### 13.4 Insights Executivos

Síntese analítica gerada dinamicamente a partir dos dados processados.

In [ ]:
receita_total = df_pivot["Receita líquida"].sum()
margem_media  = df_pivot["margem_liquida"].mean() * 100
pico_periodo  = df_pivot["Receita líquida"].idxmax()
debt_status   = "abaixo" if avg_debt < 1.5 else "acima"
alavancagem   = "baixa alavancagem" if avg_debt < 1.5 else "alavancagem elevada"
margem_status = "eficiência" if margem_media > 20 else "pressão"

SEP = "─" * 64

print(SEP)
print("  INSIGHTS EXECUTIVOS")
print(SEP)
print(f"""
  1. RECEITA
     Receita líquida acumulada de R$ {receita_total/1e9:.1f}bn no período 2023–2024,
     com pico registrado em {pico_periodo}.

  2. RENTABILIDADE
     Margem líquida média de {margem_media:.1f}%, indicando {margem_status}
     operacional consistente no período analisado.

  3. ENDIVIDAMENTO
     Debt ratio médio de {avg_debt:.2f} — {debt_status} da referência setorial
     de 1.5 — sugerindo {alavancagem} relativa ao patrimônio.

  4. ANOMALIAS
     {n_alertas} evento(s) sinalizado(s) na receita líquida.
     {"Nenhuma inconsistência contábil identificada." if n_alertas == 0
      else "Períodos sinalizados merecem investigação adicional."}
""")
print(SEP)

## 14. SQL Analytics Avançado

Aplicamos window functions para extrair insights diretamente no SQL,
sem depender de transformações em pandas.

Essa abordagem replica o padrão utilizado em ambientes corporativos
com bancos de dados relacionais (PostgreSQL, SQL Server, BigQuery).

### 14.1 Variação período a período (LAG)

Calculamos a variação absoluta e percentual da receita líquida
em relação ao período anterior usando a função LAG(),
sem nenhuma transformação fora do SQL.

In [ ]:
query_lag = """
SELECT
    date,
    value                                                AS receita,
    LAG(value) OVER (ORDER BY date)                      AS receita_anterior,
    ROUND(value - LAG(value) OVER (ORDER BY date), 2)    AS variacao_abs,
    ROUND(
        (value - LAG(value) OVER (ORDER BY date))
        * 100.0 / LAG(value) OVER (ORDER BY date),
        2
    )                                                    AS variacao_pct
FROM financial_statements
WHERE account = 'Receita líquida'
ORDER BY date
"""

df_lag = pd.read_sql(query_lag, conn)
df_lag

### 14.2 Ranking de períodos por receita (RANK)

Ranqueamos todos os períodos da série histórica pela receita líquida,
do maior para o menor, usando RANK() e DENSE_RANK().

A diferença entre as duas funções fica evidente quando há empates:
RANK() pula posições, DENSE_RANK() não.

In [ ]:
query_rank = """
SELECT
    date,
    value                                              AS receita,
    RANK()       OVER (ORDER BY value DESC)            AS rank_receita,
    DENSE_RANK() OVER (ORDER BY value DESC)            AS dense_rank_receita
FROM financial_statements
WHERE account = 'Receita líquida'
ORDER BY rank_receita
"""

df_rank = pd.read_sql(query_rank, conn)

melhor = df_rank.iloc[0]
pior   = df_rank.iloc[-1]

print(f"  Melhor período:  {melhor['date']}  —  R$ {melhor['receita']/1e9:.1f}bn")
print(f"  Pior período:    {pior['date']}  —  R$ {pior['receita']/1e9:.1f}bn")
print()

df_rank

### 14.3 Receita acumulada corrente (Running Total)

Calculamos o acumulado da receita líquida ao longo do tempo com
SUM() OVER, usando a frame clause ROWS UNBOUNDED PRECEDING.

Essa técnica é fundamental para análises de metas, projeções
anuais e curvas de crescimento.

In [ ]:
query_running = """
SELECT
    date,
    value                                              AS receita,
    ROUND(
        SUM(value) OVER (
            ORDER BY date
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ),
        2
    )                                                  AS receita_acumulada
FROM financial_statements
WHERE account = 'Receita líquida'
ORDER BY date
"""

df_running = pd.read_sql(query_running, conn)
df_running

#### Curva de receita acumulada

Visualização do crescimento acumulado ao longo dos períodos analisados.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.fill_between(
    df_running["date"],
    df_running["receita_acumulada"] / 1e9,
    alpha=0.08,
    color="black"
)
ax.plot(
    df_running["date"],
    df_running["receita_acumulada"] / 1e9,
    color="black",
    linewidth=1.5,
    marker="o",
    markersize=4
)

ax.set_title(
    "Receita Líquida Acumulada — Petrobras 2023–2024",
    fontsize=11,
    fontweight="normal"
)
ax.set_xlabel("Período")
ax.set_ylabel("Receita Acumulada (R$ bn)")
ax.spines[["top", "right"]].set_visible(False)
ax.tick_params(labelsize=8)
fig.autofmt_xdate(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
conn.close()
print("Conexão encerrada.")